# Requests, BeautifulSoup, and Data from the Web
Web data work is a useful place to practice patience. A network response is not the same thing as reliable data, HTML is not a database, and scraping becomes much safer when you think about structure, failure, and politeness from the beginning.

The requests library makes HTTP work approachable by wrapping connection details into a cleaner API. BeautifulSoup helps parse HTML and search the resulting tree structure. Together they are often used for lightweight scraping and automation.

In [1]:
import requests
from bs4 import BeautifulSoup

html_text = None
try:
    response = requests.get('https://example.com', timeout=20)
    response.raise_for_status()
    html_text = response.text
    print('status:', response.status_code)
    print('content-type:', response.headers.get('content-type'))
    print(response.text[:120])
except requests.RequestException as exc:
    print('network request could not be completed in this environment:', exc)
    html_text = '''<html><head><title>Example fallback</title></head><body><h1>Example Domain</h1><a href="https://iana.org">More information</a></body></html>'''
    print('using local fallback HTML so the parsing examples still run')

network request could not be completed in this environment: HTTPSConnectionPool(host='example.com', port=443): Max retries exceeded with url: / (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:992)')))
using local fallback HTML so the parsing examples still run


A successful HTTP status code only tells part of the story. You still need to ask whether the content type is expected, whether the page is static or heavily JavaScript-driven, and whether the site allows the kind of access you intend.

In [2]:
soup = BeautifulSoup(html_text, 'html.parser')
print('page title:', soup.title.text.strip())
print('h1 tags:', [tag.get_text(strip=True) for tag in soup.find_all('h1')])
print('links:', [a.get('href') for a in soup.find_all('a')])

page title: Example fallback
h1 tags: ['Example Domain']
links: ['https://iana.org']


BeautifulSoup is easiest to understand as a tree-navigation tool. The raw HTML text is parsed into a structured representation, and then methods like find, find_all, CSS selectors, and attribute lookups help you navigate that tree.

In [3]:
payload = {
    'title': soup.title.text.strip(),
    'link_count': len(soup.find_all('a'))
}
print(payload)

{'title': 'Example fallback', 'link_count': 1}


Robust scraping means planning for missing elements, changed markup, timeouts, rate limits, and partial responses. A fragile scraper may work once in a notebook and fail the moment the target page shifts by one HTML class name.

In [4]:
try:
    bad = requests.get('https://httpbin.org/status/404', timeout=20)
    bad.raise_for_status()
except requests.RequestException as exc:
    print('caught request-related issue:', exc)

caught request-related issue: 404 Client Error: NOT FOUND for url: https://httpbin.org/status/404


It is healthy to teach scraping alongside etiquette. Respect robots.txt when relevant, avoid aggressive request rates, cache when possible, and remember that not all technically possible scraping is socially or legally acceptable.

## Final Takeaway
Requests and BeautifulSoup work well when you treat web data collection as structured I/O rather than a quick hack. Reliable scraping depends on HTTP awareness, HTML-tree thinking, error handling, and respectful access patterns.